# Task 1: LoRA fine-tuning of Qwen2.5-0.5B-Instruct
**Task:** plain-English reminder/meeting/call request -> fixed JSON.
**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all.
Everything (data, training, eval, failures) is in this notebook. Runtime is about 10 minutes.

In [ ]:
!pip install -q -U transformers peft datasets accelerate pandas torchao

In [ ]:
import json, random, re, torch, pandas as pd
from transformers import (AutoTokenizer, AutoModelForCausalLM, TrainingArguments,
                          Trainer, DataCollatorForSeq2Seq)
from peft import LoraConfig, get_peft_model
from datasets import Dataset

random.seed(42); torch.manual_seed(42)
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

## 1. Synthetic dataset
Train = 800 examples using only "normal" phrasing (am/pm and 24h times, plain priority words).
Test = 50 in-distribution + 50 **hard** examples with phrasing never seen in training
("noon", "half past 4 pm", "quarter to 6 pm", "ASAP", ...). Total held-out = 100 (task needs >= 50).

In [ ]:
DAYS = ["today","tomorrow","monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
TASKS = ["call the dentist","buy groceries","submit the tax form","water the plants","pick up the parcel",
         "send the invoice to Rahul","book train tickets","renew car insurance","review the pull request",
         "prepare the slides","pay the electricity bill","email the landlord","back up the laptop",
         "order the birthday cake","finish the report"]
MEETINGS = ["project kickoff","budget review","sprint planning","design sync","client demo",
            "team retro","hiring panel","quarterly planning"]
CALLS = ["Priya","Mom","the bank","Amit","the vendor","Dr. Sharma","Neha","the support team"]
PRIO = {"high":["urgent","this is important","high priority","it's critical"],
        "low":["no rush","whenever possible","low priority"], "normal":[""]}
PRIO_HARD = {"high":["ASAP","drop everything for this","top priority"],
             "low":["if you get a chance","only if there's time","not important"], "normal":[""]}

def fmt_ampm(h, m):
    suf = "am" if h < 12 else "pm"
    h12 = h % 12 or 12
    return f"{h12}{suf}" if m == 0 else f"{h12}:{m:02d}{suf}"

def make_time(hard):
    if hard:
        style = random.choice(["noon","midnight","half","quarter"])
        if style == "noon": return "noon", "12:00"
        if style == "midnight": return "midnight", "00:00"
        h = random.randint(1, 11); pm = random.random() < 0.6
        h24 = h + 12 if pm else h
        suf = "pm" if pm else "am"
        if style == "half": return f"half past {h} {suf}", f"{h24:02d}:30"
        return f"quarter to {h} {suf}", f"{(h24-1)%24:02d}:45"
    h = random.randint(6, 22); m = random.choice([0, 0, 15, 30, 45])
    txt = fmt_ampm(h, m) if random.random() < 0.5 else f"{h:02d}:{m:02d}"
    return txt, f"{h:02d}:{m:02d}"

def make_example(hard=False):
    typ = random.choice(["reminder","meeting","call"])
    day = random.choice(DAYS)
    dtext = day if day in ("today","tomorrow") else f"on {day}"
    if random.random() < 0.3: dtext = dtext.title() if " " not in dtext else "on " + day.capitalize()
    ttxt, tval = make_time(hard)
    prio = random.choices(["normal","high","low"], [.5,.25,.25])[0]
    phr = random.choice((PRIO_HARD if hard else PRIO)[prio])
    if typ == "reminder":
        x = random.choice(TASKS)
        t = random.choice(["Remind me to {x} {d} at {t}","Set a reminder: {x}, {d} {t}",
                           "{d} at {t} remind me to {x}","Can you remind me to {x} {d} around {t}?"])
    elif typ == "meeting":
        x = random.choice(MEETINGS)
        t = random.choice(["Schedule a {x} meeting {d} at {t}","Set up the {x} {d} at {t}",
                           "Put {x} on my calendar {d} at {t}"])
    else:
        x = random.choice(CALLS)
        t = random.choice(["Call {x} {d} at {t}","Schedule a call with {x} {d} at {t}",
                           "Book a call with {x} {d} at {t}"])
    text = t.format(x=x, d=dtext, t=ttxt)
    if phr: text = text.rstrip("?.") + ", " + phr
    target = json.dumps({"type":typ,"title":x,"day":day,"time":tval,"priority":prio}, separators=(",",":"))
    return {"text": text, "target": target}

def gen_unique(n, hard, exclude):
    out, seen = [], set(exclude)
    while len(out) < n:
        ex = make_example(hard)
        if ex["text"] not in seen:
            seen.add(ex["text"]); out.append(ex)
    return out

train = gen_unique(800, False, [])
test_easy = gen_unique(50, False, [e["text"] for e in train])
test_hard = gen_unique(50, True, [e["text"] for e in train + test_easy])
print(train[0]); print(test_hard[0])

{'text': 'Call Neha Tomorrow at 2pm, urgent', 'target': '{"type":"call","title":"Neha","day":"tomorrow","time":"14:00","priority":"high"}'}
{'text': 'Schedule a call with Neha on Thursday at noon', 'target': '{"type":"call","title":"Neha","day":"thursday","time":"12:00","priority":"normal"}'}


In [ ]:
SYSTEM = ("Convert the user's request into JSON with exactly these keys: type (reminder|meeting|call), "
          "title (string), day (today|tomorrow|monday..sunday, lowercase), time (24-hour HH:MM), "
          "priority (low|normal|high). Output only the JSON.")

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to("cuda")

def prompt_of(text):
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build(ex):
    p = tok(prompt_of(ex["text"]), add_special_tokens=False)["input_ids"]
    t = tok(ex["target"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    return {"input_ids": p + t, "attention_mask": [1]*(len(p)+len(t)), "labels": [-100]*len(p) + t}

train_ds = Dataset.from_list(train).map(build, remove_columns=["text","target"])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

## 2. LoRA configuration (and why)
| Setting | Value | Why |
|---|---|---|
| Rank r | 16 | Task is narrow (fixed schema); low rank is enough. 8-16 is the usual range; 16 gives headroom for the time-format rules without much cost. |
| Alpha | 32 | Scaling = alpha/r = 2, a common default that lets lr ~2e-4 work without tuning. |
| Target modules | q,k,v,o,gate,up,down proj | Adapting attention AND MLP layers learns output formats/mappings better than attention-only, for a small parameter increase (~1-2%). |
| Dropout | 0.05 | Light regularisation on a synthetic dataset with repeated templates. |
| Learning rate | 2e-4 (cosine, 5% warmup) | LoRA tolerates higher lr than full fine-tuning (1e-4 to 3e-4) because only small adapters train. |
| Epochs | 2 | 800 examples x 2 epochs = 100 optimiser steps at effective batch 16. Loss plateaus quickly; more epochs mainly memorise templates. |

In [ ]:
cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                 target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, cfg)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


## 3. Evaluation helpers + BASE model score
Metrics: **valid JSON %** (strict `json.loads`), **exact match %** (all 5 fields correct, no extra keys,
after extracting the first `{...}` so we don't punish harmless markdown fences), and per-field accuracy.
The base model is evaluated with adapters disabled, same prompt, greedy decoding.

In [ ]:
KEYS = ["type","title","day","time","priority"]

@torch.no_grad()
def generate(exs, bs=25, max_new_tokens=80):
    model.eval(); outs = []
    prompts = [prompt_of(e["text"]) for e in exs]
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i+bs], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs

def parse(s):
    try: return json.loads(s.strip())
    except Exception: return None

def extract(s):
    m = re.search(r"\{.*?\}", s, re.S)
    return parse(m.group(0)) if m else None

def score(outs, exs):
    n = len(exs); strict = exact = 0; field = {k: 0 for k in KEYS}
    for o, e in zip(outs, exs):
        g = json.loads(e["target"])
        strict += parse(o) is not None
        p = extract(o)
        if isinstance(p, dict):
            for k in KEYS: field[k] += p.get(k) == g[k]
            exact += all(p.get(k) == g[k] for k in KEYS) and set(p) == set(KEYS)
    r = {"valid_json_%": 100*strict/n, "exact_match_%": 100*exact/n}
    r.update({f"{k}_acc_%": 100*v/n for k, v in field.items()})
    return r

results, raw = {}, {}
with model.disable_adapter():
    for name, ds in [("easy", test_easy), ("hard", test_hard)]:
        raw[("base", name)] = generate(ds)
        results[("base", name)] = score(raw[("base", name)], ds)
print(pd.DataFrame(results).T.round(1))

           valid_json_%  exact_match_%  type_acc_%  title_acc_%  day_acc_%  \
base easy          92.0            0.0        80.0          2.0       48.0   
     hard          86.0            0.0        76.0          4.0       42.0   

           time_acc_%  priority_acc_%  
base easy        82.0            48.0  
     hard        12.0            20.0  


## 4. Train

In [ ]:
bf16 = torch.cuda.is_bf16_supported()
args = TrainingArguments(output_dir="out", per_device_train_batch_size=8, gradient_accumulation_steps=2,
                         num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine",
                         warmup_steps=5, logging_steps=10, save_strategy="no",
                         fp16=not bf16, bf16=bf16, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
trainer.train()

Step,Training Loss
10,0.157365
20,0.014328
30,0.004118
40,0.000555
50,0.000461
60,0.000277
70,0.000129
80,0.000131
90,0.000152
100,0.000116


TrainOutput(global_step=100, training_loss=0.017763150122482328, metrics={'train_runtime': 236.4655, 'train_samples_per_second': 6.766, 'train_steps_per_second': 0.423, 'total_flos': 433822281553920.0, 'train_loss': 0.017763150122482328, 'epoch': 2.0})

## 5. Fine-tuned score vs base (this is your results table)

In [ ]:
for name, ds in [("easy", test_easy), ("hard", test_hard)]:
    raw[("ft", name)] = generate(ds)
    results[("ft", name)] = score(raw[("ft", name)], ds)
table = pd.DataFrame(results).T.round(1)
table.index.names = ["model", "test_set"]
print(table)

                valid_json_%  exact_match_%  type_acc_%  title_acc_%  \
model test_set                                                         
base  easy              92.0            0.0        80.0          2.0   
      hard              86.0            0.0        76.0          4.0   
ft    easy             100.0          100.0       100.0        100.0   
      hard             100.0           42.0       100.0        100.0   

                day_acc_%  time_acc_%  priority_acc_%  
model test_set                                         
base  easy           48.0        82.0            48.0  
      hard           42.0        12.0            20.0  
ft    easy          100.0       100.0           100.0  
      hard          100.0        56.0            80.0  


## 6. Failure analysis: pick 3 for your report

In [ ]:
fails = []
for name, ds in [("easy", test_easy), ("hard", test_hard)]:
    for o, e in zip(raw[("ft", name)], ds):
        p = extract(o); g = json.loads(e["target"])
        if not (isinstance(p, dict) and all(p.get(k) == g[k] for k in KEYS)):
            fails.append({"set": name, "input": e["text"], "gold": e["target"], "pred": o.strip()})
print(len(fails), "failures")
for f in fails[:10]:
    print(f"[{f['set']}] {f['input']}\n  gold: {f['gold']}\n  pred: {f['pred']}\n")

29 failures
[hard] Set up the design sync on friday at quarter to 6 pm
  gold: {"type":"meeting","title":"design sync","day":"friday","time":"17:45","priority":"normal"}
  pred: {"type":"meeting","title":"design sync","day":"friday","time":"18:00","priority":"normal"}

[hard] Tomorrow at noon remind me to water the plants, drop everything for this
  gold: {"type":"reminder","title":"water the plants","day":"tomorrow","time":"12:00","priority":"high"}
  pred: {"type":"reminder","title":"water the plants","day":"tomorrow","time":"12:00","priority":"normal"}

[hard] Schedule a client demo meeting on wednesday at half past 4 am, not important
  gold: {"type":"meeting","title":"client demo","day":"wednesday","time":"04:30","priority":"low"}
  pred: {"type":"meeting","title":"client demo","day":"wednesday","time":"06:00","priority":"low"}

[hard] Put quarterly planning on my calendar on wednesday at quarter to 7 am, if you get a chance
  gold: {"type":"meeting","title":"quarterly planning","

**How to explain failures:** look for the pattern. Likely causes: (1) phrasing never seen in training
("quarter to", "half past" need arithmetic, not just copying), (2) am/pm -> 24h conversion errors,
(3) priority phrases outside the training vocabulary ("ASAP", "if you get a chance"), (4) a 0.5B model's limited
arithmetic. Check whether your failures are mostly on the `hard` set. If so, that is the evidence.

In [ ]:
model.save_pretrained("lora_adapter")
table.to_csv("task1_results.csv"); json.dump(fails, open("task1_failures.json","w"), indent=1)
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))
from google.colab import files
for f in ["task1_results.csv", "task1_failures.json"]: files.download(f)

trainable params: 8798208


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>